# 03 - MedViT (+ Supervised Contrastive Learning)

**Owner:** Viet  |  **Plane:** sagittal  |  **Sweep task:** acl

Plain MedViT (hybrid CNN + transformer) for the sweep, plus a **MedViT + SupCon**
run that happens regardless of the sweep winner (transformers benefit most from
contrastive pretraining). CBAM can attach to MedViT's conv stages if MedViT wins.
See `Project Pipeline.md` -> sections 2, 3.2, 3.3.

## How to use
1. Run the **Colab bootstrap** cell, then the **data-staging** cell (copies the sagittal plane to local disk for a big T4 speedup).
2. Implement the MedViT branch in `src/model_factory.py` and the SupCon pieces in `src/contrastive_learning.py`.
3. Train on the sagittal ACL task; report AUC for plain MedViT and MedViT + SupCon.

**T4 optimizations baked in:** mixed precision (AMP) is on by default in `fit` / `pretrain_encoder` / `train_linear_classifier`; data loaders use `pin_memory` + non-blocking transfers; and data is staged on local disk (`DATA_ROOT`).

> **GPU equivalent:** `for-gpu/run_medvit.py` (plain / augmentation / SupCon sections, Stages 1-2). The strong-augmentation fine-tune compared in the report is `for-gpu/run_cnn.py --backbone medvit`.

In [ ]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import os
# Reduce CUDA fragmentation on the T4 (must be set before any CUDA use).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys
from pathlib import Path

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # Team shared drive (edit if your path differs).
    CODES_DIR = Path('/content/drive/Shareddrives/MRNet_Group3/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder."
)

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Auto-reload edited src/*.py without restarting the kernel.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload unavailable ({e}); restart kernel after editing src/*.py')

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK - data folder found.')


In [ ]:
# ============================================================
# (T4 speedup) Stage the data on local disk once per session.
# Reading .npy from mounted Google Drive is slow and is the main
# bottleneck on Colab. Copying the sagittal plane to /content
# (local SSD) makes every epoch's data loading much faster.
# We pass DATA_ROOT into build_dataloaders / get_pos_weight below.
# ============================================================
import shutil
from pathlib import Path

PLANE = config.DEFAULT_PLANE  # 'sagittal'

if IN_COLAB:
    drive_data = Path(config.DATA_DIR)
    local_data = Path('/content/data')
    if not local_data.exists():
        print(f'Copying {PLANE} plane from Drive to {local_data} (one-time)...')
        for split in ['train', 'valid']:
            src = drive_data / split / PLANE
            if src.exists():
                dst = local_data / split / PLANE
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copytree(src, dst)
        for csv in drive_data.glob('*.csv'):
            shutil.copy(csv, local_data / csv.name)
        print('Done.')
    DATA_ROOT = str(local_data)
else:
    # Local run: read straight from the resolved data folder.
    DATA_ROOT = str(config.DATA_DIR)

print('DATA_ROOT:', DATA_ROOT)


In [ ]:
# Sanity check: MedViT backbone returns (B, feat_dim) with the head removed.
from src.model_factory import build_backbone
import torch

backbone, feat_dim = build_backbone("medvit", pretrained=True)
print("feat_dim:", feat_dim)
print("output:", backbone(torch.randn(4, 3, 224, 224)).shape)  # expect (4, 1024)


## Plain MedViT transfer-learning (sagittal, ACL)

**Feature-extraction** transfer learning: the ImageNet-pretrained MedViT backbone
is **frozen** and we train only the slice-attention pool + linear head. This is
fast and fits the T4 (no backprop through MedViT). To fine-tune the backbone
end-to-end instead, set `freeze_backbone=False` (slower; uses gradient
checkpointing to fit memory).

Each exam is one batch (variable #slices), so we use **gradient accumulation** for
a larger effective batch size. Slices are center-cropped to **224** to match
MedViT's pretrained input. Class imbalance is handled by a global
`pos_weight = neg/pos` in `BCEWithLogitsLoss`.

In [ ]:
import torch
from src.data_pipeline import build_dataloaders, get_pos_weight, set_seed
from src.model_factory import build_model
from src.training_utils import fit, validate, EarlyStopping

set_seed(config.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

TASK = "acl"

# MedViT pretrained at 224 -> crop MRNet's 256 slices to 224.
# root_dir=DATA_ROOT reads from local disk on Colab (see the data-copy cell).
train_loader, valid_loader = build_dataloaders(
    root_dir=DATA_ROOT,
    task=TASK,
    plane=config.DEFAULT_PLANE,
    train_augment="light",
    batch_size=1,
    num_workers=2,
    output_size=224,
)

# Class imbalance handled by a single global pos_weight = neg/pos (train split).
pos_weight = get_pos_weight(root_dir=DATA_ROOT, task=TASK, train=True).to(device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", pos_weight.item())

# freeze_backbone=True -> feature-extraction: MedViT stays fixed, we only train
# the slice-attention pool + linear head. Much faster (no backbone backward) and
# fits the T4 comfortably. Drop to freeze_backbone=False later to fine-tune.
model = build_model(backbone="medvit", pretrained=True, num_classes=1,
                    dropout=0.3, freeze_backbone=True)
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,            # higher LR: we're training a fresh head, not fine-tuning
    weight_decay=1e-4,
)

best_state, history = fit(
    model, train_loader, valid_loader, optimizer, device,
    num_epochs=15,
    criterion=criterion,
    accumulation_steps=8,   # effective batch size of 8 exams
    early_stopping=EarlyStopping(patience=5, mode="max"),
    checkpoint_path=str(config.CHECKPOINTS_DIR / "medvit_acl.pt"),
)

print("Plain MedViT — best validation:", validate(model, valid_loader, device))

## MedViT with different augmentation methods (comparison)

Train MedViT under each augmentation preset (`none`, `light`, `medium`, `strong`)
and compare validation AUC. Everything else (architecture, optimizer, `pos_weight`,
seed) is held fixed, so augmentation strength is the only variable. This trains one
model per preset — lower `num_epochs` if you're short on time.

In [ ]:
import pandas as pd

aug_results = {}
for preset in ["none", "light", "medium", "strong"]:
    set_seed(config.SEED)   # same init + data order for a fair comparison
    tl, vl = build_dataloaders(
        root_dir=DATA_ROOT, task=TASK, plane=config.DEFAULT_PLANE,
        train_augment=preset, batch_size=1, num_workers=2, output_size=224,
    )
    m = build_model(backbone="medvit", pretrained=True, num_classes=1,
                    dropout=0.3, freeze_backbone=True)
    opt = torch.optim.Adam(
        [p for p in m.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4
    )
    fit(
        m, tl, vl, opt, device,
        num_epochs=10, criterion=criterion, accumulation_steps=8,
        early_stopping=EarlyStopping(patience=4, mode="max"),
        checkpoint_path=str(config.CHECKPOINTS_DIR / f"medvit_acl_{preset}.pt"),
        verbose=False,
    )
    aug_results[preset] = validate(m, vl, device)
    print(preset, aug_results[preset])

pd.DataFrame(aug_results).T   # rows = preset, cols = loss / auc / accuracy

## MedViT + Supervised Contrastive Learning

Two stages on a **fresh** MedViT encoder, reusing the same loaders:
1. **Pretrain** the encoder (backbone + slice pooling) with the supervised
   contrastive loss on exam embeddings.
2. **Freeze** the encoder and train a linear classifier head on top.

If you hit GPU out-of-memory in stage 1, lower `supcon_batch` (e.g. 4) — it
controls how many exam embeddings are held for each contrastive step.

In [ ]:
from src.model_factory import build_model
from src.contrastive_learning import pretrain_encoder, train_linear_classifier
from src.training_utils import validate

# Fresh MedViT encoder for the SupCon ablation.
encoder = build_model(backbone="medvit", pretrained=True, num_classes=1, dropout=0.0)

# Stage 1: supervised contrastive pretraining.
encoder, supcon_hist = pretrain_encoder(
    encoder, train_loader,
    epochs=10, supcon_batch=8, temperature=0.07, lr=1e-4, device=device,
)

# Stage 2: freeze encoder, train a linear classifier head.
encoder, probe_hist = train_linear_classifier(
    encoder, train_loader, valid_loader,
    epochs=15, lr=1e-3, criterion=criterion, device=device,
)

print("MedViT + SupCon — validation:", validate(encoder, valid_loader, device))